# Préparation des données

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

from fraud_detection.data.load import PROJECT_ROOT, load_interim

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def save(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=120, bbox_inches="tight")

In [ ]:
df = load_interim()

# Vue d'ensemble

## Forme de la base de données

In [ ]:
print(df.shape)
print(f"{df.memory_usage(deep=True).sum() / 1e9:.2f} Go")
df["isFraud"].value_counts()

In [ ]:
df.head(30)

In [ ]:
cols = df.columns.tolist()
print(cols)

In [ ]:
import re
from collections import defaultdict

families = defaultdict(list)
for col in df.columns:
    prefix = re.match(r"[A-Za-z_]+?(?=_?\d|$)", col).group()
    families[prefix].append(col)

for prefix, cols in families.items():
    print(f"{prefix} ({len(cols)}): {cols[:5]}{' ...' if len(cols) > 5 else ''}")

## Indicateur temporel

In [ ]:
print(df["TransactionDT"].min(), df["TransactionDT"].max())
print(df["TransactionDT"].max() / 86400, "jours si l'unité est la seconde")

On commentce par faire l'hypothèse que l'unité de TransactionDT est en secondes, par rapport à un temps t que l'on ne connaît pas et qui est notre origine de référence. La première transaction démarrerait donc 1 jour après t.

In [ ]:
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["hour"].value_counts().sort_index().plot(
    kind="bar", figsize=(10, 3), title="Transactions par heure"
)

On remarque ici qu'il y'a un creux dans la répartition des transactions par heure. Cela nous aide à confirmer que TransactionDT est bien un décalage temporel en secondes par rapport à un moment t choisi arbitrairement.

In [ ]:
df["day"] = df["TransactionDT"] // 86400
daily = df.groupby("day")["isFraud"].agg(["size", "mean"])

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
daily["size"].plot(ax=axes[0], title="Nombre de transactions par jour")
daily["mean"].plot(ax=axes[1], title="Taux de fraude par jour")
daily["mean"].rolling(7, center=True).mean().plot(ax=axes[1], color="red", linewidth=2)
save("temps")

# Valeurs manquantes

## Vérification générale

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print((missing > 0.9).sum(), "colonnes à plus de 90 % de manquants")
print((missing == 0).sum(), "colonnes complètes")

groups = {"V": r"^V\d+$", "C": r"^C\d+$", "D": r"^D\d+$", "M": r"^M\d+$", "id": r"^id_"}
for name, pattern in groups.items():
    cols = df.filter(regex=pattern).columns
    print(
        f"{name}: {len(cols)} colonnes, {df[cols].isna().mean().mean():.0%} de manquants en moyenne"
    )

In [ ]:
has_identity = df.filter(regex=r"^id_").notna().any(axis=1)
df.groupby(has_identity)["isFraud"].agg(["size", "mean"])

## Vérifications du lien entre identité et type de produit

In [ ]:
pd.crosstab(df["ProductCD"], has_identity)

In [ ]:
pd.crosstab(df["ProductCD"], has_identity, values=df["isFraud"], aggfunc="mean")

## Test sur le type de carte : lien avec id et type de produit

In [ ]:
pd.crosstab(df["card4"], has_identity)

In [ ]:
pd.crosstab(df["card4"], has_identity, values=df["isFraud"], aggfunc="mean")

In [ ]:
pd.crosstab(df["card4"], df["ProductCD"], values=df["isFraud"], aggfunc="mean").round(3)

In [ ]:
pd.crosstab(df["card4"], df["ProductCD"])

## Colonnes "V"

In [ ]:
v_cols = df.filter(regex=r"^V\d+$").columns
na_counts = df[v_cols].isna().sum()
blocks = na_counts.groupby(na_counts).apply(lambda s: list(s.index))
print(len(blocks), "blocs de colonnes V partageant le même nombre de manquants")
blocks.apply(len).sort_values(ascending=False).head(10)

In [ ]:
def select_uncorrelated(cols, threshold=0.9, sample=50_000):
    corr = df[cols].sample(sample, random_state=0).corr().abs()
    kept = []
    for col in cols:
        if all(corr.loc[col, k] < threshold or pd.isna(corr.loc[col, k]) for k in kept):
            kept.append(col)
    return kept

In [ ]:
for threshold in [0.95, 0.9, 0.75]:
    kept = [c for block in blocks for c in select_uncorrelated(block, threshold)]
    print(f"Seuil {threshold} : {len(kept)} colonnes V conservées sur {len(v_cols)}")

## Colonnes avec plus de 90% de manquants

In [ ]:
for col in missing[missing > 0.9].index:
    present = df[col].notna()
    print(
        f"{col:8s} renseignée : {present.mean():5.1%} | "
        f"fraude si renseignée : {df.loc[present, 'isFraud'].mean():.3f} | "
        f"sinon : {df.loc[~present, 'isFraud'].mean():.3f}"
    )

In [ ]:
pd.crosstab(df["ProductCD"], df["dist2"].notna())

In [ ]:
pd.crosstab(
    df["ProductCD"], df["dist2"].notna(), values=df["isFraud"], aggfunc="mean"
).round(3)

In [ ]:
sub = df[has_identity]
for col in ["id_07", "id_18", "id_24"]:
    present = sub[col].notna()
    print(
        f"{col}: fraude si renseignée {sub.loc[present, 'isFraud'].mean():.3f} | "
        f"sinon {sub.loc[~present, 'isFraud'].mean():.3f}"
    )